# Model Klasifikasi Aspek Baseline (Ulasan Aplikasi)
Implementasi model baseline Multi-label Aspect Classification menggunakan TF-IDF Vectorizer dan Support Vector Machine (SVM).

In [ ]:
import os
import json
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, f1_score, accuracy_score
import joblib

## 1. Load Data

In [ ]:
dataset_path = '../data/cleaned_app_reviews_with_aspect_labels.csv'
df = pd.read_csv(dataset_path).dropna(subset=['text_clean'])
print(f"Total data: {len(df)} baris")

## 2. Split Data Train / Test

In [ ]:
aspect_cols = ['aspek_rasa', 'aspek_harga', 'aspek_aplikasi', 'aspek_pelayanan', 'aspek_kecepatan', 'aspek_stok', 'aspek_pengiriman', 'aspek_pesanan']

X = df['text_clean']
y = df[aspect_cols].values

# Split data 80% Train, 20% Test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42
)
print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

## 3. Vektorisasi TF-IDF

In [ ]:
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)
print(f"TF-IDF feature size: {X_train_tfidf.shape[1]}")

## 4. Pelatihan Model Multi-label SVM
Menggunakan OneVsRestClassifier pembungkus SVM linear untuk menangani multi-label targets.

In [ ]:
base_svm = SVC(kernel='linear', probability=True, random_state=42)
multilabel_model = OneVsRestClassifier(base_svm)

print("Training model SVM...")
multilabel_model.fit(X_train_tfidf, y_train)
print("Selesai.")

## 5. Evaluasi Model

In [ ]:
y_pred = multilabel_model.predict(X_test_tfidf)

micro_f1 = f1_score(y_test, y_pred, average='micro')
macro_f1 = f1_score(y_test, y_pred, average='macro')

print(f"Micro F1-Score: {micro_f1:.4f}")
print(f"Macro F1-Score: {macro_f1:.4f}\n")

# Detail per aspek
for i, col in enumerate(aspect_cols):
    print(f"=== Klasifikasi Aspek: {col} ===")
    print(classification_report(y_test[:, i], y_pred[:, i]))
    print("-" * 50)

## 6. Simpan Model & Evaluasi

In [ ]:
os.makedirs('../weights', exist_ok=True)
os.makedirs('../../reports', exist_ok=True)

joblib.dump(multilabel_model, '../weights/aspect_classifier_svm.pkl')
joblib.dump(vectorizer, '../weights/tfidf_vectorizer_aspect.pkl')

eval_dict = {
    "model_type": "TF-IDF + OneVsRest SVM (App Reviews)",
    "overall_micro_f1": float(micro_f1),
    "overall_macro_f1": float(macro_f1),
    "aspects_evaluated": aspect_cols
}

with open('../../reports/aspect_model_evaluation_app.json', 'w') as f:
    json.dump(eval_dict, f, indent=4)
print("Model dan evaluasi baseline berhasil disimpan.")